### Trader Execution

Walk-forward is the test design: train on the past, test on the next unseen stretch, with a hard fenced split so nothing leaks across. This chapter runs the machine-learning workflow on Binance 1-hour spot data, drawn survivorship-complete from the `data.binance.vision` archive: the full historical USDT universe (612 pairs, roughly a third of them delisted), point-in-time screened per bar rather than the survivor-only active list.

The work runs in stages: build and label the 1h dataset, screen the candidate features (elastic-net / glmnet path), fit logistic regression, random forest and LightGBM head-to-head under a 60/40 confidence filter, assess and tune on blind data to nominate a best model, then test stability against buy-and-hold, a coin flip, and a triple-Supertrend rules baseline before any capital is risked.

All metrics are compiled in `outputs/AA-evals/`, comparing the features and ATR-scaled label from `inputs/build_dataset_1h.py`, the final-year split from `inputs/train_model_1h.py`, the per-model scoring from `inputs/train_model.py`, and the four-metric evaluation from `inputs/eval_report.py`. Run it from the project `.venv`, which holds `pandas-ta` and `TA-Lib`.

### Supertrend Integration

The triple-Supertrend work added this cycle lives in three places in this chapter:
- **Features** (`### Feature Variables`): `inputs/build_dataset_1h.py` `supertrend_block()` emits the `f_st_` family - three ATR-channel bands voting, signed band distances, the reversal flip, EMA-200 distance - ported from the live bot `inputs/supertrend.py`. Activates on the next dataset rebuild.
- **Baseline** (`### Stability`): `inputs/baseline_supertrend_1h.py` trades the bot's long-only rule on the out-of-sample year and scores it after fees through ClaudeTrader's metrics - the rules benchmark the model must beat.
- **Engine**: ClaudeTrader installed editable into the `.venv` supplies the risk and performance modules behind the baseline.

Chapter One frames the Supertrend as a trend vote; Chapter Two as an ATR exit/sizing control. Full record: `tasks/integration-2026-06-23-claudetrader-supertrend.md`.

### Environment

This notebook must run on the project .venv kernel "Python (day-trader .venv)", which holds joblib, scikit-learn, lightgbm, ccxt, pandas-ta and TA-Lib. In Jupyter you do NOT "activate" a venv in a shell, you select its kernel: Kernel menu >> Change Kernel >> "Python (day-trader .venv)". If that kernel is not listed, register it once from a terminal, then reopen the notebook:

```
   /Volumes/PortableSSD/Github/day-trader/.venv/bin/python -m ipykernel install --user \
       --name day-trader --display-name "Python (day-trader .venv)"
```

In [1]:
import os, sys
from pathlib import Path


if sys.version_info[:2] < (3, 11):
    raise RuntimeError(f"Use the project .venv (Python 3.12); kernel is {sys.version.split()[0]}.")
try:
    import numpy as np, pandas as pd, joblib
    import matplotlib.pyplot as plt
except ModuleNotFoundError as e:
    raise RuntimeError(
        f"Missing '{e.name}'. This kernel is not the project .venv (running {sys.executable}). "
        "Switch to 'Python (day-trader .venv)' via Kernel -> Change Kernel, then re-run."
    ) from e

# Shared modeling + evaluation code: the SAME functions the scripts run.
INPUTS_DIR = None
for cand in ["inputs", os.path.join("..", "inputs")]:
    if os.path.isdir(cand):
        INPUTS_DIR = os.path.abspath(cand)
        sys.path.insert(0, INPUTS_DIR); break
REPO_ROOT = Path(INPUTS_DIR).parent          # repo root = parent of inputs/, regardless of cwd
import build_dataset_1h as bd        # 1h features, ATR-scaled label, point-in-time screen
import train_model as tm             # shared: build_models, evaluate, confidence_filtered, costs
import train_model_1h as t1          # 1h load + final-year split
import eval_report
HAVE_LGBM = tm.HAVE_LGBM

def _layer(mod):
    try:
        __import__(mod); return "yes"
    except Exception:
        return "no"

OUTPUTS = REPO_ROOT / "outputs"                          # absolute <repo>/outputs, never the cwd-relative one
MODEL_DIR = OUTPUTS / "3B-model-training"; MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"outputs -> {OUTPUTS}")
print(f"env ready  .  kernel: {sys.executable}")
print(f"python {sys.version.split()[0]}  .  lightgbm {'yes' if HAVE_LGBM else 'no'}"
      f"  .  pandas-ta {_layer('pandas_ta')}  .  TA-Lib {_layer('talib')}")

# --- inline table rendering (Positron-safe) -------------------------------------------
# Positron routes a bare display(df) to its live Data Explorer pane, which shows blank
# inline and dies on reload. show_table() emits a static HTML table that renders in the
# cell and survives reload, in Positron, Jupyter and VS Code alike.
from IPython.display import HTML, display, Markdown  # noqa: F401  (Markdown used later)

def show_table(x):
    obj = x.to_frame() if (hasattr(x, "to_frame") and getattr(x, "ndim", 2) == 1) else x
    display(HTML(obj.to_html()))

outputs -> /Volumes/PortableSSD/Github/day-trader/outputs
env ready  .  kernel: /Volumes/PortableSSD/Github/day-trader/.venv/bin/python
python 3.12.13  .  lightgbm yes  .  pandas-ta yes  .  TA-Lib yes


### Data Preparation

Everything between the raw exchange archives and the first model lives here. The operations are defined once in `inputs/build_dataset_1h.py` and `inputs/train_model_1h.py`; the cells here load the prepared data and list the predictor families. Acquisition (dead coins included), profiling and the split are owned by the Survivorship Pipeline section below and documented in `tasks/data-pipeline-methodology.md`.

### Dataset Label

One row per coin per hour across the full USDT spot market, each row kept only if that coin was tradable at that hour (the point-in-time screen). Predictors are scale-invariant (ratios and distances, so one model fits a \$60k coin and a \$0.20 coin). The label is an ATR-scaled triple barrier on a short day-trade horizon: a win is a +2 ATR rise before a -1 ATR drop, within 48 hours.

### Data Acquisition

The cleaning, the point-in-time screen, the survivorship correction and the forward-chained split are the subject of the Survivorship Pipeline below, each with its own diagnostic. Stage A enumerates the historical universe from the archive so delisted coins are present; Stage B profiles coverage, gaps, breadth and liquidity and derives the usable start, minimum history and purge/embargo; Stage C splits forward in time with that embargo and point-in-time per-fold universes.

### Split Audit

We split by time on purpose, so we cannot force class proportions; instead we audit whether the temporal, multi-coin split is representative, on three fronts.

1. Panel composition. The pool mixes hundreds of coins with unequal history (BTC to 2017, newer coins about two years), so the model is implicitly weighted toward long-history coins and the regimes they lived through. Check the row share per coin in train and test, the per-coin base rate on each side, and which coins appear in only one window.

2. Temporal drift. Because the split is by time, the hold-out can be a different regime, so a NO-GO with large drift means regime change, not absence of edge. Compare the label rate and each feature's distribution between the training years and the test year: base-rate shift (two-proportion z-test), continuous drift (Kolmogorov-Smirnov plus a ranked Population Stability Index, flag PSI above 0.10 and 0.25), and proportion shift on the binary features. Re-assert that the embargo separates the last training label from the first test bar.

3. Label imbalance. The barrier label is roughly 0.32 positive. We act on calibrated probabilities above a 0.60 threshold rather than argmax, and `class_weight="balanced"` distorts exactly those probabilities, so class-weighting and SMOTE (fit strictly within the training fold) are candidates graded on the after-fee metric and on calibration, not defaults. Any resampling stays inside the training window; cross-validation is embargoed and time-indexed, never shuffled.

Cohen's Kappa, per-class precision, recall and F1, the confusion matrix and OOB are diagnostics for whether the minority class is learned; they do not decide the outcome. The after-fee Metric 2 (net expectancy per confident trade, against buy-and-hold and a coin-flip) remains the deciding number.

### Next Steps

The cell below surfaces the active knobs (split size, trade filter, costs, label), read-only from the scripts. Then Import Data loads the prepared dataset and Feature Variables lists the candidate families. Acquisition, profiling and splitting are exercised in the Survivorship Pipeline section; modelling follows from Variable Selection.

In [2]:
# Knobs live in the scripts so the notebook cannot drift. Read-only here.
CONFIG = dict(
    oos_days     = t1.OOS_DAYS,         # final year held out of sample
    embargo_days = t1.EMBARGO_DAYS,     # = label horizon in days
    conf_hi      = tm.CONF_HI,          # confidence filter act-long (Keller Metric 1)
    conf_lo      = tm.CONF_LO,          # act-short / stand-aside
    cost_pct     = tm.COST_PCT,         # round-trip drag (fee + slippage), Metric 2
)
lab = bd.LABEL
print(f"label: +{lab['tgt_atr']} ATR before -{lab['stp_atr']} ATR within {lab['horizon_bars']} bars "
      f"({lab['horizon_bars']//bd.BARS_PER_DAY}d)  .  hold out final {CONFIG['oos_days']}d, "
      f"embargo +/-{CONFIG['embargo_days']}d  .  confidence {CONFIG['conf_lo']}-{CONFIG['conf_hi']}  .  "
      f"cost {CONFIG['cost_pct']:.2f}% round trip")

label: +2.0 ATR before -1.0 ATR within 48 bars (2d)  .  hold out final 365d, embargo +/-2d  .  confidence 0.4-0.6  .  cost 0.20% round trip


### Import Data

Loads the prepared 1h dataset at `inputs/binance-data/dataset_1h_allmarket.parquet` (via `bd.DATASET_PATH`), built offline from the `data.binance.vision` archives by `inputs/build_dataset_1h.py`; `t1.load` reads it and keeps only the point-in-time `in_sample` rows. If it has not been built, the cell prints the build command. The cleaning, screen and label that produced this file are described once in Data Preparation above, so they are not repeated here.

The build is incremental and runs on demand via `tasks/run_auto_eval.sh`; it can also run nightly from cron, pulling the latest 24 hours, rebuilding the dataset, and retraining, so the load always reflects the most recent full day:

```bash
# 02:00 daily: refresh data -> rebuild dataset -> retrain
0 2 * * *  cd /Volumes/PortableSSD/Github/day-trader && \
  .venv/bin/python inputs/binance-data/flow_data.py --interval 1h --all-market && \
  .venv/bin/python inputs/build_dataset_1h.py && \
  .venv/bin/python inputs/train_model_1h.py
```

The cell below is reference-only: it reads the live build and reports when each file was produced, so the provenance is always visible. The trade-flow feature is `flow_imbalance = 2 * (taker_buy_base / volume) - 1`, the signed share of each bar's volume lifting the ask, in [-1, +1].

In [3]:
# Import Data: build provenance (how/when the data was built + the build config) AND the load +
# characteristics, in one cell. Reads the LIVE config from build_dataset_1h and the file
# timestamps; does NOT rebuild anything.
import os, datetime as _dt

def _built(p):
    base = os.path.splitext(p)[0]
    for ext in (".parquet", ".csv"):
        f = base + ext
        if os.path.exists(f):
            return (f"{os.path.basename(f)}  "
                    f"{_dt.datetime.fromtimestamp(os.path.getmtime(f)):%Y-%m-%d %H:%M}, "
                    f"{os.path.getsize(f)/1e6:.0f} MB")
    return "NOT BUILT YET"

# --- provenance + build configuration (nothing loaded yet) ---
print("SOURCE / LOCATION   (Binance spot, 1-hour bars, data.binance.vision archives)")
print(f"  klines root : {bd.DEFAULT_KLINES_ROOT}")
print(f"  flow table  : {_built(bd.DEFAULT_FLOW)}")
print(f"  dataset     : {_built(bd.DATASET_PATH)}")
print("  storage     : Parquet (columnar, compressed, dtype-preserving); CSV is the fallback")
print("\nTIME WINDOWS  (counts of bars; 1 bar = 1 hour)")
print(f"  wall-clock (daily x24): {bd.WC}")
print(f"  intraday              : {bd.HR}")
print(f"\nLABEL  (ATR-scaled triple barrier): {bd.LABEL}")
print(f"SCREEN gates                       : {bd.SCREEN}")
print(f"DATA-QUALITY gate                  : {bd.DATA_QUALITY}")
print("\nTo (re)build offline from the .venv (reference - not run here):")
print("  .venv/bin/python inputs/binance-data/flow_data.py --interval 1h --all-market")
print("  .venv/bin/python inputs/build_dataset_1h.py")

# --- load the prepared dataset + report its characteristics ---
# bd.read_frame prefers the Parquet file (real dtypes kept, ~2.6x smaller, loads in seconds);
# it falls back to the CSV if only that exists.
raw = bd.read_frame(bd.DATASET_PATH)
if raw is not None:
    raw = raw.sort_values("datetime").reset_index(drop=True)
    feat = bd.feature_columns(raw)
    insamp = raw["in_sample"] if "in_sample" in raw.columns else pd.Series(True, index=raw.index)
    df = raw[insamp].reset_index(drop=True)        # model on the point-in-time in-sample rows
    print(f"\nLOADED {len(raw):,} rows from Parquet")
    print(f"  in-sample rows     : {len(df):,}  ({len(df)/len(raw):.1%} of total)")
    print(f"  coins              : {raw['symbol'].nunique()}")
    print(f"  features           : {len(feat)}")
    print(f"  date range         : {raw['datetime'].min()}  ->  {raw['datetime'].max()}")
    print(f"  base rate all / in-sample : {raw['label'].mean():.3f} / {df['label'].mean():.3f}")
    fams = [("wall-clock", "f_wc_"), ("intraday", "f_hr_"), ("in-house TA", "f_ta_"),
            ("pandas-ta", "f_ta_pta_"), ("TA-Lib", "f_tl_"), ("flow", "f_flow_")]
    parts = []
    for nm, pre in fams:
        if pre == "f_ta_":
            cols = [c for c in feat if c.startswith("f_ta_") and not c.startswith("f_ta_pta_")]
        else:
            cols = [c for c in feat if c.startswith(pre)]
        parts.append(f"{nm} {len(cols)}")
    print("  feature families   : " + ", ".join(parts))
    print(f"  in-sample rows with any NaN feature: {int(df[feat].isna().any(axis=1).sum()):,}")
    print("  rows per coin:")
    show_table(raw["symbol"].value_counts().rename("rows").to_frame())
else:
    df = None; feat = []
    print("\n1h dataset not built yet. Build it from the project .venv:\n"
          "  .venv/bin/python inputs/build_dataset_1h.py\n"
          "(or a coin subset:  .venv/bin/python inputs/build_dataset_1h.py -s BTCUSDT ETHUSDT ...)")

SOURCE / LOCATION   (Binance spot, 1-hour bars, data.binance.vision archives)
  klines root : /Volumes/PortableSSD/Github/day-trader/inputs/binance-data/klines_1h
  flow table  : flow_1h.parquet  2026-06-21 10:36, 458 MB
  dataset     : dataset_1h_allmarket.parquet  2026-06-21 11:15, 684 MB
  storage     : Parquet (columnar, compressed, dtype-preserving); CSV is the fallback

TIME WINDOWS  (counts of bars; 1 bar = 1 hour)
  wall-clock (daily x24): {'ema_fast': 336, 'ema_mid': 2184, 'ema_slow': 3000, 'rsi': 336, 'bb': 336, 'bb_std': 2.0, 'atr': 336, 'rv_short': 168, 'rv_long': 720, 'vol': 480, 'mom': [120, 240, 480, 1440]}
  intraday              : {'ema_fast': 12, 'ema_mid': 26, 'ema_slow': 50, 'rsi': 14, 'bb': 20, 'bb_std': 2.0, 'atr': 14, 'rv_short': 24, 'rv_long': 168, 'vol': 24, 'mom': [6, 12, 24, 72, 168]}

LABEL  (ATR-scaled triple barrier): {'tgt_atr': 2.0, 'stp_atr': 1.0, 'horizon_bars': 48, 'atr_len': 14}
SCREEN gates                       : {'min_quote_volume_usdt': 30000000,

,rows
symbol,
BTC/USDT,75229
ETH/USDT,75229
BNB/USDT,73293
LTC/USDT,72405
ADA/USDT,69437
LINK/USDT,62715
FET/USDT,61837
DASH/USDT,61177
DOGE/USDT,58971


### Feature Variables

The candidate set is broad by design; the variable selection that follows (elastic-net / glmnet paths) prunes it. It spans two window families (wall-clock = the daily windows x24, and a shorter intraday family), an in-house extra-indicator block (Williams %R, Stochastic, CCI, CMF, MFI, ADX/DMI, Aroon), a triple-Supertrend block (`f_st_`: three ATR-channel bands voting, signed band distances, the reversal flip, EMA-200 distance, ported from the live bot), the trade-flow imbalance, and - from the `.venv` - optional pandas-ta (PPO, TRIX, Vortex, CMO, Fisher, Chande Kroll Stop) and TA-Lib (SAR, MAMA, Ultimate Oscillator, Hilbert cycle features, candlestick patterns) layers. All causal and scale-invariant.

**Variables in the code cell below** - this cell just counts and summarises the predictors; nothing new is built:

| variable | what it is, in plain terms | type / where it comes from | what it does here |
| --- | --- | --- | --- |
| `feat` | the names of every predictor column the model may use | `list[str]`, from the Import Data cell (`bd.feature_columns`) | the full candidate set being summarised |
| `df` | the in-sample dataset (the rows kept as tradable) | `DataFrame`, from the Import Data cell | source of the summary statistics |
| `fam` | the feature families paired with their column-name prefixes (e.g. wall-clock -> `f_wc_`) | `list` of (name, prefix) | groups the predictors so each family can be counted |
| `pre` | one family's column-name prefix | `str` (loop variable) | picks out that family's columns |
| `cols` | the predictor columns that belong to one family | `list[str]` | counted, to show how many features each family contributes |

In [4]:
if df is not None:
    fam = [("wall-clock (wc)", "f_wc_"), ("intraday (hr)", "f_hr_"),
           ("in-house TA", "f_ta_"), ("pandas-ta", "f_ta_pta_"),
           ("TA-Lib", "f_tl_"), ("flow", "f_flow_")]
    for name, pre in fam:
        if pre == "f_ta_":
            cols = [c for c in feat if c.startswith("f_ta_") and not c.startswith("f_ta_pta_")]
        else:
            cols = [c for c in feat if c.startswith(pre)]
        print(f"  {name:16s}: {len(cols)}")
    print(f"  {'TOTAL':16s}: {len(feat)}")
    show_table(df[feat].describe().T[["mean", "std", "min", "max"]].round(3))

  wall-clock (wc) : 13
  intraday (hr)   : 14
  in-house TA     : 8
  pandas-ta       : 7
  TA-Lib          : 15
  flow            : 4
  TOTAL           : 61


,mean,std,min,max
f_wc_ema_fast_mid,-0.018,0.194,-1.729,0.655
f_wc_ema_mid_slow,-0.006,0.064,-0.566,0.153
f_wc_rsi,0.502,0.026,0.372,0.647
f_wc_bb_pos,0.531,0.358,-1.825,2.594
f_wc_atr_pct,0.014,0.006,0.002,0.076
f_wc_rv_short,0.009,0.004,0.002,0.049
f_wc_rv_long,0.010,0.004,0.002,0.055
f_wc_rv_ratio,0.957,0.247,0.257,2.037
f_wc_vol_ratio,1.032,1.123,0.001,129.313
f_wc_mom_120,0.011,0.097,-0.716,0.801


In [ ]:
# --- Exploratory data analysis on the built 1h dataset (read-only) -----------------------
# Five views beyond the family counts above: label balance over time, per-coin base rates and
# history, feature redundancy (top correlated pairs), the most label-discriminative features,
# and a correlation heatmap. All on the in-sample rows the model actually trains on.
if df is not None and feat:
    # 1) label balance overall + by calendar year (is the +2/-1 ATR barrier stable across regimes?)
    yr = (df.assign(year=df["datetime"].dt.year).groupby("year")
          .agg(rows=("label", "size"), base_rate=("label", "mean")).round(3))
    print(f"label base rate (in-sample): {df['label'].mean():.3f} on {len(df):,} rows, "
          f"{df['symbol'].nunique()} coins, {df['datetime'].min().date()} -> {df['datetime'].max().date()}")
    print("by year:")
    show_table(yr)

    # 2) per-coin base rate + span + panel share (which coins dominate the fit?)
    g = df.groupby("symbol")
    per_coin = pd.DataFrame({
        "rows": g.size(),
        "row_share": (g.size() / len(df)).round(3),
        "base_rate": g["label"].mean().round(3),
        "first": g["datetime"].min().dt.date.astype(str),
        "last": g["datetime"].max().dt.date.astype(str),
    }).sort_values("rows", ascending=False)
    print(f"\ntop 10 coins carry {per_coin['row_share'].head(10).sum():.0%} of in-sample rows "
          f"(the panel tilts to long-history coins):")
    show_table(per_coin.head(10))

    # 3) feature redundancy: most correlated pairs (candidates the L1 path below will prune)
    corr = df[feat].corr()
    cc = corr.where(~np.eye(len(feat), dtype=bool)).abs().unstack().dropna()
    top_pairs = cc.sort_values(ascending=False)[::2].head(12).round(3).reset_index()
    top_pairs.columns = ["feature_a", "feature_b", "abs_corr"]
    print("\nmost correlated feature pairs (redundancy the variable-selection step prunes):")
    show_table(top_pairs)

    # 4) most label-discriminative features: standardized gap between label 1 and 0
    m1, m0 = df[df.label == 1][feat].mean(), df[df.label == 0][feat].mean()
    sep = ((m1 - m0) / df[feat].std().replace(0, np.nan)).abs().sort_values(ascending=False).round(3)
    print("\ntop 12 features by standardized class separation  |mean(1) - mean(0)| / sd:")
    show_table(sep.head(12).rename("separation").to_frame())

    # 5) correlation heatmap over the most label-separating features (visual structure)
    top_feats = sep.head(18).index.tolist()
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(df[top_feats].corr(), cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(top_feats))); ax.set_xticklabels(top_feats, rotation=90, fontsize=7)
    ax.set_yticks(range(len(top_feats))); ax.set_yticklabels(top_feats, fontsize=7)
    ax.set_title("Correlation of the most label-separating features")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); fig.tight_layout()
    (OUTPUTS / "PNG").mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUTS / "PNG" / "3-eda-feature-correlation.png", dpi=150)
    plt.show()
else:
    print("build the 1h dataset first (see Import Data).")

### Survivorship Pipeline

The dataset is only as honest as the panel it is built on. Three dependent stages, run in order, make that panel defensible. Full rationale and the parameter decisions live in `tasks/data-pipeline-methodology.md`.

- **Stage A - acquire dead coins.** A universe built from `exchangeInfo` (live, TRADING only) silently drops every delisted pair and overstates performance. `inputs/acquire_vision.py` enumerates the historical universe from the `data.binance.vision` archive listing instead (612 USDT pairs, ~31% delisted), so the graveyard is included; downloads are checksum-verified and resumable.
- **Stage B - profile the panel.** `inputs/profile_panel.py` measures coverage, gaps, listing/delisting timeline, breadth, survivorship share and liquidity, then derives the split parameters (usable start, minimum history, purge/embargo) from those diagnostics.
- **Stage C - split forward in time.** `inputs/wf_splitter.py` is a forward-chained walk-forward splitter with purge/embargo at every cut and point-in-time per-fold universes - the regime-stability companion to the final-year OOS in `train_model_1h`.

### Stage A

In [5]:
# Stage A: prove the historical universe includes dead coins. Crawl is ~4s, no download here.
# (inputs/ is already on sys.path from the Environment cell.)
import acquire_vision as av

crawl = av.crawl_archive_symbols(quote='USDT', verbose=False)
uni = crawl['symbols']
print(f"historical USDT universe (incl. dead coins): {len(uni)} pairs "
      f"(+{len(crawl['leveraged'])} leveraged excluded)")

# Stage A.5 guard: known-delisted pairs MUST be present, else the list came from a live endpoint.
aud = av.audit_dead_coins(uni)

# Survivorship share vs a dated active snapshot (writes inputs/binance-data/exchange_info_snapshot_<date>.json).
active, snap_path = av.snapshot_exchange_info()
part = av.partition_survivorship(uni, active)
print(f"survivorship: {part['n_universe']} universe / {part['n_active']} active / "
      f"{part['n_delisted']} delisted = {part['delisted_share']:.1%} dead coins")
# The heavy pull (operator's Mac):  python inputs/acquire_vision.py download --interval 1h

historical USDT universe (incl. dead coins): 612 pairs (+52 leveraged excluded)
dead-coin audit: 10/10 known-delisted present in universe
snapshot: 433 active USDT pairs as of 2026-06-23 -> /Volumes/PortableSSD/Github/day-trader/inputs/binance-data/exchange_info_snapshot_2026-06-23.json
survivorship: 612 universe / 424 active / 188 delisted = 30.7% dead coins


### Stage B

In [6]:
# Stage B: panel profiling. Loads the persisted profile; BUILDS it on first run if absent
# (writes run-stamped artefacts under outputs/3A-training-test-data/panel-profile/). Rebuild with:
#     python inputs/profile_panel.py
# Absolute paths come from the module, so this is independent of the notebook's cwd.
import json, os, pandas as pd
import profile_panel as pp

latest = os.path.join(pp.PROFILE_ROOT, 'latest.txt')
if not os.path.exists(latest):
    print('No Stage B profile found -- building over the full klines_1h panel.')
    print('First run only; it streams every coin twice, so expect a few minutes...')
    pp.profile(root=pp.DEFAULT_KLINES_ROOT, out_root=pp.PROFILE_ROOT, make_plot=False)

run = open(latest).read().strip()
md_tbl = pd.read_parquet(os.path.join(run, 'symbol_metadata.parquet'))
dec = json.load(open(os.path.join(run, 'decision_summary.json')))
print('profile run  :', run)
print('symbols      :', len(md_tbl), '| delisted share:', f"{dec['survivorship']['delisted_share']:.1%}")
print('usable start :', dec['usable_start_date'])
print('min history  :', dec['min_history_days'], 'days')
print('purge/embargo:', dec['purge_embargo_days'], 'days', f"({dec['purge_embargo_bars']} bars)")
print('point-of-entry:', dec['point_of_entry_rule'][:90], '...')
display(md_tbl[['symbol', 'first_bar', 'last_bar', 'n_bars', 'coverage', 'max_gap_hours',
                'halt_flag', 'active']].head(10))

No Stage B profile found -- building over the full klines_1h panel.
First run only; it streams every coin twice, so expect a few minutes...
profiling 384 symbols from /Volumes/PortableSSD/Github/day-trader/inputs/binance-data/klines_1h
  profiled 50/384 symbols...
  profiled 100/384 symbols...
  profiled 150/384 symbols...
  profiled 200/384 symbols...
  profiled 250/384 symbols...
  profiled 300/384 symbols...
  profiled 350/384 symbols...
  monthly QV 50/384...
  monthly QV 100/384...
  monthly QV 150/384...
  monthly QV 200/384...
  monthly QV 250/384...
  monthly QV 300/384...
  monthly QV 350/384...

survivorship: 384 symbols, 0 delisted (0.0%), dead bars 0.0%
usable start: 2019-08-01 | min history: 157d | purge/embargo: 125d
artefacts -> /Volumes/PortableSSD/Github/day-trader/outputs/3A-training-test-data/panel-profile/2026-06-23T072849Z
profile run  : /Volumes/PortableSSD/Github/day-trader/outputs/3A-training-test-data/panel-profile/2026-06-23T072849Z
symbols      : 384 | delist

,symbol,first_bar,last_bar,n_bars,coverage,max_gap_hours,halt_flag,active
0,BTCUSDT,2017-08-17 04:00:00+00:00,2026-06-20 23:00:00+00:00,77389,0.9984,75,True,True
1,ETHUSDT,2017-08-17 04:00:00+00:00,2026-06-20 23:00:00+00:00,77389,0.9984,75,True,True
2,BNBUSDT,2017-11-06 03:00:00+00:00,2026-06-20 23:00:00+00:00,75453,0.9984,75,True,True
3,NEOUSDT,2017-11-20 03:00:00+00:00,2026-06-20 23:00:00+00:00,75117,0.9984,75,True,True
4,LTCUSDT,2017-12-13 03:00:00+00:00,2026-06-20 23:00:00+00:00,74565,0.9984,75,True,True
5,QTUMUSDT,2018-03-19 08:00:00+00:00,2026-06-20 23:00:00+00:00,72289,0.9988,10,False,True
6,ADAUSDT,2018-04-17 04:00:00+00:00,2026-06-20 23:00:00+00:00,71597,0.9988,10,False,True
7,TUSDUSDT,2018-05-31 09:00:00+00:00,2026-06-20 23:00:00+00:00,66541,0.9422,3995,True,True
8,IOTAUSDT,2018-05-31 09:00:00+00:00,2026-06-20 23:00:00+00:00,70536,0.9988,10,False,True
9,ONTUSDT,2018-06-08 07:00:00+00:00,2026-06-20 23:00:00+00:00,70356,0.9989,10,False,True


### Stage C

In [7]:
# Stage C: the splitter draws usable_start + per-fold universes from the Stage B run above.
# Reuses `bd` (build_dataset_1h) from the Environment cell.
import wf_splitter as wf
print(wf.METHODOLOGY)
pb, lf, lh = wf.purge_embargo_bars()
print(f'purge/embargo = max(feature lookback {lf}, label horizon {lh}) = {pb} bars '
      f'= {wf.purge_embargo_days()} days')

df_wf = bd.read_frame(bd.DATASET_PATH)
if df_wf is not None:
    if 'in_sample' in df_wf.columns:
        df_wf = df_wf[df_wf['in_sample']].copy()
    sp = wf.WalkForwardSplitter(n_folds=5, test_days=120, scheme='expanding',
                                usable_start=wf.load_usable_start(),
                                universes=wf.load_universes())
    display(sp.fold_table(df_wf))   # per-fold spans + coin counts (C.5 composition audit)
    # Regime-stability metrics (fits a model per fold; run on the .venv):
    # wf.evaluate_walkforward(df_wf, bd.feature_columns(df_wf), sp)
else:
    print('No dataset yet at', bd.DATASET_PATH)

## Splitting methodology: forward-chained, not stratified

Objective: estimate out-of-sample performance under REGIME DRIFT, not interpolation within a
known distribution. The data-generating process drifts across time -- the 2017-18 ICO mania,
the 2021 bull run, the 2022 deleveraging, and the current regime are different processes. A
split that interleaves test bars among training bars borrows the future to predict the past
and yields an error estimate that will not survive live trading.

Stratified sampling is rejected. Stratifying on variance forces similar class proportions
across splits by drawing observations regardless of position in time, which destroys the
temporal ordering that is the whole object of interest. It answers "can the model interpolate
within a known distribution"; the question that matters is "can it extrapolate forward into an
unknown regime." Only a forward-chained split measures that.

Design: expanding/rolling walk-forward on shared calendar boundaries; purge

,fold,train_start,train_end,test_start,test_end,n_train,n_test,n_coins,embargo_days
0,0,2019-08-01,2024-06-23,2024-10-26,2025-02-23,271088,28742,21,125
1,1,2019-08-01,2024-10-21,2025-02-23,2025-06-23,294746,25482,19,125
2,2,2019-08-01,2025-02-18,2025-06-23,2025-10-21,329261,31086,23,125
3,3,2019-08-01,2025-06-18,2025-10-21,2026-02-18,356443,24993,21,125
4,4,2019-08-01,2025-10-16,2026-02-18,2026-06-18,387233,16801,20,125


### Variable Selection

Stage ii: prune the broad candidate set to the subset that earns its place, BEFORE the head-to-head and before any hyperparameter tuning, on the TRAINING window only so the final-year hold-out stays blind (otherwise the blind score is no longer blind).

This is wired to `inputs/variable_selection.py`, a native-Python re-implementation of the R `glmnet` + `coefplot` elastic-net screening idiom, applied to the `f_` feature set and the binary triple-barrier `label` (`family="binomial"`). The R study was the recipe only; no forestry variables are used.

| R (glmnet / coefplot) | Python (`variable_selection.py`) | what it gives |
| --- | --- | --- |
| `cv.glmnet(alpha=1, nfolds=10)` | `enet_cv(l1_ratio=1.0, n_folds=10)` | k-fold CV over a glmnet-style lambda grid; lambda.min / lambda.1se |
| `plot(cv.glmnet)` | `plot_cv_curve()` | binomial deviance vs log-lambda, 1-se whiskers, nonzero counts |
| `coefplot::coefpath` | `plot_coefpath()` / `plot_coefpath_interactive()` | coefficient trajectories (PNG + interactive HTML with a range slider) |
| `coefplot::coefplot(sort="magnitude")` | `plot_coef_ci()` | survivors refit unpenalized (statsmodels Logit) for a 95% CI dot-whisker |
| variables kept at `lambda.1se` | `screen()` | the screened subset, into `SELECTED_FEATURES` |

**Why lasso, why the 1-se rule.** The L1 path drives weak coefficients to exactly zero, so the survivors are a genuine subset, not a re-weighting. `lambda.1se`, the most-regularized lambda within one standard error of the minimum-deviance lambda, is the conventional parsimonious choice: it trades a sliver of fit for a smaller, more stable feature set. Selection runs on TRAIN only. The saga logistic path is slow at the least-regularized end, so the cell samples 25k training rows, which is plenty for the screening picture. Set `feat = SELECTED_FEATURES` and re-run training to fit the head-to-head on the screened subset.

In [ ]:
# Stage ii: elastic-net (glmnet-analogue) variable selection on the TRAINING split ONLY, via
# inputs/variable_selection.py end-to-end: build_matrix -> enet_cv -> CV curve + coef paths ->
# screen at lambda.1se -> refit survivors for 95% CIs. The blind final year is never touched.
import importlib, variable_selection as vs
importlib.reload(vs)
from IPython.display import Image, display

VARSEL_DIR = OUTPUTS / "AA-evals" / "varselect"
VARSEL_DIR.mkdir(parents=True, exist_ok=True)

if df is not None and feat:
    tr_vs, _te_vs, _cut = t1.split(df)                          # select on TRAIN; OOS stays blind
    samp = tr_vs.dropna(subset=[*feat, "label"])
    if len(samp) > 25000:                                       # saga path is slow; 25k screens fine
        samp = samp.sample(25000, random_state=0)
    X, y, _ = vs.build_matrix(samp, y_col="label", x_cols=feat, standardize=True)
    res = vs.enet_cv(X, y, family="binomial", l1_ratio=1.0, n_folds=10, verbose=False)
    print(f"lasso path on {len(samp):,} TRAIN rows x {len(feat)} features.")
    print(f"lambda.min={res['lambda_min']:.4g} (nonzero {int(res['nonzero'][res['i_min']])}), "
          f"lambda.1se={res['lambda_1se']:.4g} (nonzero {int(res['nonzero'][res['i_1se']])})")

    p_cv  = vs.plot_cv_curve(res, str(VARSEL_DIR / "cv_curve.png"))
    p_cp  = vs.plot_coefpath(res, str(VARSEL_DIR / "coefpath.png"))
    p_cph = vs.plot_coefpath_interactive(res, str(VARSEL_DIR / "coefpath.html"))
    display(Image(filename=p_cv)); display(Image(filename=p_cp))

    kept = vs.screen(res, "1se") or vs.screen(res, "min")        # survivors, largest |coef| first
    SELECTED_FEATURES = [n for n, _ in kept]
    ci_cols = (SELECTED_FEATURES or [n for n, _ in vs.screen(res, "min")])[:12]
    p_ci = vs.plot_coef_ci(samp, "binomial", str(VARSEL_DIR / "coef_ci.png"),
                           y_col="label", x_cols=ci_cols,
                           title="Screened coefficients (logit, 95% CI)")
    display(Image(filename=p_ci))

    print(f"\nelastic-net kept {len(SELECTED_FEATURES)} of {len(feat)} features at lambda.1se.")
    print("top kept:", ", ".join(f"{n}({v:+.3f})" for n, v in kept[:15]))
    print("interactive coefficient path (open in browser):", p_cph)
    print("\nTo train the head-to-head on this subset: set  feat = SELECTED_FEATURES  then re-run training.")
else:
    SELECTED_FEATURES = []
    print("build the 1h dataset first (see Import Data).")

### Training Regime

The split is executed here (in the cell just below), then the models are trained on all history before the final-year cut and scored once on the held-out year; the representativeness and imbalance audit for this split is described in Data Preparation above. Three models compete: logistic regression, random forest, and LightGBM (Tier 1). Each is reported at the 0.5 threshold and under the 60/40 confidence filter (Keller Metric 1); read precision against the base rate, which sits well below 0.5. The full record - Metric 1, Metric 2 (P&L after the 0.20% cost), and Metric 3 (AUC by volatility regime) - is written to `outputs/AA-evals/` by `eval_report.write_comparison`, tagged "head-to-head (1h)", so it sits beside the other runs in `evaluation-scores.md`.

In [9]:
# Split and scoring come from the scripts: t1.split for the 1h final-year hold-out; tm.evaluate,
# tm.build_models, tm.confidence_filtered shared with inputs/train_model.py. Nothing reimplemented.
train, test, cut = t1.split(df)
Xtr, ytr, Xte, yte = train[feat], train["label"], test[feat], test["label"]
base_te = yte.mean()
span = lambda d: f"{d['datetime'].min().date()} to {d['datetime'].max().date()}"
print(f"train {len(train):,} ({span(train)})  .  test {len(test):,} ({span(test)})  .  "
      f"cut {pd.Timestamp(cut).date()}  embargo +/-{t1.EMBARGO_DAYS}d  .  base rate {base_te:.3f}")

train 381,207 (2017-12-15 to 2025-06-16)  .  test 76,789 (2025-06-19 to 2026-06-18)  .  cut 2025-06-18  embargo +/-2d  .  base rate 0.303


In [10]:
# Fast interactive pass. The full-fidelity 3-model run on ALL history is inputs/train_model_1h.py
# (script / auto-eval). The heavy part here is evaluate()'s 5-fold TimeSeriesSplit CV on each model
# (3 models x (5 CV fits + 1 final fit) on ~381k rows = minutes). For the notebook we fit a bounded
# sample taken across the whole period and re-sorted by time, so the time-series CV stays
# chronological. Raise TRAIN_CAP or set it to None for the full run (slow).
TRAIN_CAP = 80_000
if TRAIN_CAP is None or len(train) <= TRAIN_CAP:
    tr_fit = train
else:
    tr_fit = train.sample(TRAIN_CAP, random_state=0).sort_values("datetime")
Xtr_f, ytr_f = tr_fit[feat], tr_fit["label"]
print(f"fitting on {len(tr_fit):,} of {len(train):,} train rows "
      f"({tr_fit['datetime'].min().date()} to {tr_fit['datetime'].max().date()})"
      + ("  [sampled for the notebook; full run is inputs/train_model_1h.py]"
         if (TRAIN_CAP and len(train) > TRAIN_CAP) else ""))

lines, scored = [], []
for name, mdl in tm.build_models(HAVE_LGBM):
    prec, m = tm.evaluate(name, mdl, Xtr_f, ytr_f, Xte, yte, base_te, lines)
    scored.append((prec, name, mdl, m))
print("\n".join(lines))
_, best_name, best_model, best = max(scored, key=lambda t: t[0])
print(f"\nchosen: {best_name}  (precision {best['prec']:.3f} vs base {base_te:.3f}, AUC {best['auc']:.3f})")

fitting on 80,000 of 381,207 train rows (2017-12-16 to 2025-06-16)  [sampled for the notebook; full run is inputs/train_model_1h.py]

--- LogisticRegression ---
  train CV ROC-AUC (5-fold TimeSeriesSplit): 0.528
  test accuracy : 0.484
  test precision(buy): 0.307   (base rate 0.303)
  test recall(buy)   : 0.560
  test ROC-AUC       : 0.504
  confusion matrix [rows=true 0/1, cols=pred 0/1]:
   [[24164, 29352], [10241, 13032]]
  precision lift over base rate: +0.004
  confidence filter (act if p>=0.60 or p<=0.40): keeps 2.7% of test rows (n=2075)
    precision(buy) 0.285  recall 0.663  F1 0.398  (base rate 0.303)

--- RandomForest ---
  train CV ROC-AUC (5-fold TimeSeriesSplit): 0.526
  test accuracy : 0.489
  test precision(buy): 0.310   (base rate 0.303)
  test recall(buy)   : 0.557
  test ROC-AUC       : 0.509
  confusion matrix [rows=true 0/1, cols=pred 0/1]:
   [[24627, 28889], [10319, 12954]]
  precision lift over base rate: +0.007
  confidence filter (act if p>=0.60 or p<=0.40): 

In [11]:
# Honesty gate: precision must clearly beat the base rate and AUC clear 0.55. It does not
# trade; Metric 2 (next cell) is the money test.
margin = best["prec"] - base_te
go = (best["prec"] > base_te + 0.05) and (best["auc"] > 0.55)
verdict = "GO" if go else "NO-GO"
print("HONESTY GATE:", "GO (edge survives out-of-sample)" if go
      else "NO-GO (no demonstrable edge - do not trade)")
MODEL_DIR.mkdir(parents=True, exist_ok=True)   # self-sufficient: works even if the setup cell was not re-run
joblib.dump({"model": best_model, "features": feat, "name": best_name,
             "trained_through": str(pd.Timestamp(cut).date()),
             "test_base_rate": float(base_te), "go": bool(go)}, MODEL_DIR / "model_1h.joblib")
summary = (f"{best_name}: test precision(buy)={best['prec']:.3f} base_rate={base_te:.3f} "
           f"lift={margin:+.3f} AUC={best['auc']:.3f} recall={best['rec']:.3f} acc={best['acc']:.3f} -> {verdict}")
(MODEL_DIR / "model_metrics_1h.txt").write_text(summary + "\n\n" + "\n".join(lines) + "\n")
print("saved", MODEL_DIR / "model_1h.joblib", "and model_metrics_1h.txt")

HONESTY GATE: NO-GO (no demonstrable edge - do not trade)
saved /Volumes/PortableSSD/Github/day-trader/outputs/3B-model-training/model_1h.joblib and model_metrics_1h.txt


In [12]:
# Metrics 2 (P&L after costs) and 3 (regime-stratified AUC) plus AA-evals bookkeeping, from
# inputs/eval_report.py - the same record the script writes. Appends one row to
# outputs/AA-evals/evaluation-scores.md (+ .pdf, .docx), tagged "head-to-head (1h)".
imp = getattr(best_model, "feature_importances_", None)
regime = "f_wc_rv_long" if "f_wc_rv_long" in test.columns else next((c for c in feat if "rv_long" in c), None)
meta = dict(dataset_rows=len(df), n_features=len(feat),
            train_rows=len(train), test_rows=len(test), base_rate=float(base_te),
            cut=str(pd.Timestamp(cut).date()), embargo=t1.EMBARGO_DAYS,
            conf_hi=tm.CONF_HI, conf_lo=tm.CONF_LO, chosen=best_name, verdict=verdict,
            eval_type="head-to-head (1h)",
            dataset_label=f"{len(df):,}r / {len(feat)}f (1h all-market)",
            fi_names=list(feat) if imp is not None else None,
            fi_values=[float(v) for v in imp] if imp is not None else None,
            regime_vol=test[regime].tolist() if regime else None,
            trade_ret=test["trade_ret"].tolist() if "trade_ret" in test.columns else None,
            cost_pct=tm.COST_PCT)
results = [m for (_, _, _, m) in scored]
rec = eval_report.write_comparison(str(OUTPUTS / "AA-evals"), results, yte, meta)
print("evaluation record:", rec["md"])

evaluation record: /Volumes/PortableSSD/Github/day-trader/outputs/AA-evals/2026-06-23/eval-head-to-head-20260623.md


In [13]:
# Teaching display: the strongest tree model's top features. The AA-evals record above
# already saves this chart; this is just an inline look.
tree = next((mdl for (_, name, mdl, _) in scored
             if name in ("LightGBM", "RandomForest")), None)
imp = getattr(tree, "feature_importances_", None) if tree is not None else None
if imp is not None:
    order = np.argsort(imp)[::-1][:15]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh([feat[i] for i in order][::-1], imp[order][::-1], color="#0B3D66")
    ax.set_title(f"{tree.__class__.__name__}: top 15 feature importances")
    fig.tight_layout()
    (OUTPUTS / "PNG").mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUTS / "PNG" / "3B-feature-importance.png", dpi=160)
    plt.show()

### Model Tuning

Two sweeps, both scored on the after-fee out-of-sample scoreboard and logged to the consolidated `outputs/AA-evals/evaluation-scores.md`:

- **Priority 1b - label geometry** (`inputs/sweep_label_1h.py`): the ATR-scaled triple barrier swept over (target, stop, horizon). Each cell reports its own base rate, the breakeven win rate stop/(stop+target), and net P&L/trade after cost, so a lopsided geometry is obvious.
- **Priority 1a - exit geometry** (`inputs/walkforward.py`): stop and take-profit, plus per-coin trailing stops and a time-decaying take-profit - the same question from the exit side.

Settle the two together, then unify the forked label/exit configs. The broad feature set is left broad here on purpose; the variable-selection pass (likelihood-ratio tests, elastic-net paths) prunes predictors before final fitting.

In [17]:
# The sweeps are full-market and run as scripts from the project .venv:
#   .venv/bin/python inputs/sweep_label_1h.py    # Priority 1b: label geometry (target, stop, horizon)
#   .venv/bin/python inputs/walkforward.py       # Priority 1a: exit geometry (stop, take-profit, trail)
# Both append to the consolidated scoreboard; here we render that one table so tuning reads off it.
#from IPython.display import Markdown, display
#scoreboard = OUTPUTS / "AA-evals" / "evaluation-scores.md"
#if scoreboard.exists():
#    display(Markdown(scoreboard.read_text()))
#else:
#    print("no evaluation-scores.md yet - run a model (cells above) or a sweep script first.")
# Quick inline 1b probe (small grid; the full grid is the script above):
#   import sweep_label_1h as sw
#   coins, scols = sw.precompute(bd.DEFAULT_KLINES_ROOT, bd.DEFAULT_FLOW_CSV, None)
#   sw.score_cell(coins, scols, 3.0, 1.0, 48, tm.COST_PCT/100.0, tm.CONF_HI)

### Model Assessment

A head-to-head scorecard in the style of a caret model-assessment table. For each model it reports the tuned hyperparameters, the in-sample (Full) error, the cross-validated error, and their ratio. Errors are RMSE and MAE on the predicted probabilities - here RMSE is the square root of the Brier score, the caret-style classification RMSE. The RMSEratio (Full RMSE / CV RMSE) flags overfitting: near 1 means the model generalises; well below 1 means it fits the training data far better than it holds up out of sample. Cross-validation is time-ordered (not random folds), and the final year is kept as a single blind test. The table and the best model are logged to the consolidated scoreboard as a "tuning" row.

In [ ]:
# Caret-style assessment (stage iv). Renders the latest PERSISTED record if one exists; otherwise
# runs a quick inline assessment (logreg + lightgbm) and renders it directly, so the cell ALWAYS
# shows a result. The full zoo + ensemble on all history is the script:
#   .venv/bin/python inputs/model_assessment_1h.py
import glob
import model_assessment_1h as ma
from IPython.display import Markdown, display

recs = sorted(glob.glob(str(OUTPUTS / "AA-evals" / "*" / "model-assessment-*.md")))
if recs:
    print(f"rendering persisted record: {recs[-1]}")
    display(Markdown(open(recs[-1]).read()))
elif df is not None and feat:
    print("no persisted record yet - running a quick inline assessment (logreg + lightgbm)...")
    rows, tr_a, te_a, _cut_a = ma.assess(df, feat, only=["logreg", "lightgbm"])
    ranked = sorted(rows, key=lambda r: r["cv_rmse"])
    tbl = pd.DataFrame([{"model": r["model"], "hyperparameters": r["hp"],
                         "Full RMSE": round(r["full_rmse"], 4), "CV RMSE": round(r["cv_rmse"], 4),
                         "RMSEratio": round(r["rmse_ratio"], 3)} for r in ranked])
    show_table(tbl)
    best = ranked[0]
    blind = ma.blind_metrics(best["est"], tr_a, te_a, feat, tm.COST_PCT / 100.0, tm.CONF_HI)
    print(f"\nbest by CV RMSE: {best['model']} (RMSEratio {best['rmse_ratio']:.3f})")
    print(f"blind final-year: AUC {blind['auc']:.3f}, precision(p>=0.60) {blind['prec']:.3f} "
          f"vs base {blind['base']:.3f}, net {blind['net']*100:+.3f}%/trade on {blind['trades']:,} trades")
    print("\nPersist the full record (all models + ensemble):  .venv/bin/python inputs/model_assessment_1h.py")
else:
    print("build the 1h dataset first (see Import Data).")

### Stability

Confirm any edge is not an artifact: parameter stability, results split by market type, and three baselines - a coin flip, buy-and-hold, and the triple-Supertrend rules benchmark (`inputs/baseline_supertrend_1h.py`, scored after fees through ClaudeTrader's metrics) - plus an optional bootstrap on trade returns. Only then paper trade, then a tiny live allocation. No live trading until a configuration clearly beats all three baselines, out-of-sample and after fees.